# LSER Project Interface

This notebook is the primary user-facing interface for the cleaned LSER pipeline. It runs the package APIs directly, supports optional feature hooks, shows saved figures and KPIs, and provides both existing-molecule and new-molecule scoring workflows.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd

from LSER_Project_Cleaned import (
    lookup_molecule,
    run_full_pipeline,
    score_existing_molecules,
    score_new_molecules_direct,
    score_new_molecules_pipeline,
)
from LSER_Project_Cleaned.common import DEFAULT_FIGURES_DIR

## 1. Run the full pipeline

In [ ]:
pipeline_results = run_full_pipeline(
    save_output=True,
    save_figures=True,
    figures_dir=DEFAULT_FIGURES_DIR,
    repeated_split_tests=0,
)

clean_df = pipeline_results["data_cleaning"]["data"]
explore_df = pipeline_results["data_exploration"]["data"]
cluster_df = pipeline_results["clustering"]["data"]
training = pipeline_results["model_training"]
model_bundle = training["model_bundle"]
metrics = training["metrics"]
inference = pipeline_results["model_inference"]

print("Cleaned shape:", clean_df.shape)
print("Explored shape:", explore_df.shape)
print("Clustered shape:", cluster_df.shape)
print("Model families:", sorted(inference["predictions"].keys()))

## 2. Inspect saved figures and KPIs

In [ ]:
figure_paths = {
    "exploration": pipeline_results["data_exploration"]["artifacts"]["figure_paths"],
    "clustering": pipeline_results["clustering"]["artifacts"]["figure_paths"],
    "training": pipeline_results["model_training"]["artifacts"]["figure_paths"],
}
figure_paths

In [ ]:
pd.Series(metrics["KPI"], name="KPI")

## 3. Look up an individual molecule in the modeled dataset

In [ ]:
lookup_molecule(cluster_df, molecular_name="Diazepam").head()

In [ ]:
existing_scores = score_existing_molecules(
    model_bundle,
    molecular_name="Diazepam",
    basis_molecule="Diazepam",
)
existing_scores["predictions"]["MLR"][:1]

## 4. Change the basis molecule for relative scores

In [ ]:
basis_molecule = "Diazepam"
pipeline_results = run_full_pipeline(
    save_output=False,
    save_figures=False,
    basis_molecule=basis_molecule,
)
pipeline_results["model_inference"]["predictions"]

## 5. Optional feature hooks

Define dataframe transforms here. These hooks are applied during data cleaning before downstream stages rerun.

In [ ]:
def add_example_feature(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    if {"HBondAcceptorCount", "PolarSurfaceArea(sqAngstrom)"}.issubset(df.columns):
        df["PolarSurfaceAreaifHaccepting"] = np.where(
            df["HBondAcceptorCount"].fillna(0).to_numpy() == 0,
            0,
            df["PolarSurfaceArea(sqAngstrom)"].fillna(0).to_numpy(),
        )
    return df

In [ ]:
feature_hook_results = run_full_pipeline(
    save_output=False,
    save_figures=False,
    feature_hooks=[add_example_feature],
)
feature_hook_results["data_cleaning"]["metadata"]["applied_feature_hooks"]

## 6. Score new molecules with the direct-model path

This path assumes the dataframe already contains training-ready features or enough columns for imputation.

In [ ]:
direct_input = model_bundle["model_df"].head(2).copy()
direct_input = direct_input.drop(columns=[model_bundle["feature_columns"][0]])
direct_input.loc[:, "Mw(g/moL)"] = np.nan

direct_scores = score_new_molecules_direct(
    model_bundle,
    direct_input,
    basis_molecule="Diazepam",
)
direct_scores["predictions"]["MLR"][:2]

## 7. Score new molecules with the fuller pipeline path

This path is intended for raw-ish input and runs the new rows through the cleaning, exploration, and clustering logic before direct scoring.

In [ ]:
rawish_input = clean_df.head(2).copy()
pipeline_scores = score_new_molecules_pipeline(
    pipeline_results,
    rawish_input,
    basis_molecule="Diazepam",
    feature_hooks=[add_example_feature],
)
pipeline_scores["direct_scoring"]["predictions"]["MLR"][:2]

## 8. Load external new-molecule data

Replace this example with a dataframe that contains your new molecules.

In [ ]:
# Example placeholder
# new_molecule_df = pd.read_csv("my_new_molecules.csv")
# score_new_molecules_direct(model_bundle, new_molecule_df, basis_molecule="Diazepam")